# 섹터 데이터 수집 (morning-report ADR-003)

**목적**: ADR-003 섹터 강도 산식 구현에 필요한 데이터를 pykrx로 수집해서
Google Drive에 parquet으로 저장한다.

**수집 내역** (최근 2년):
1. `sector_map.parquet` — KRX 22업종 × 162종목 분류 매핑
2. `sector_index_weekly.parquet` — 22업종 업종지수 **주봉** OHLCV (Weinstein Stage + 6M RS)
3. `stocks_daily.parquet` — 162종목 **일봉** OHLCV (IBD 6M % + MA50 Breadth)

**사용법**:
- 런타임 → 런타임 유형 변경 → **Python 3** (GPU 불필요)
- 런타임 → **모두 실행** (Ctrl+F9)
- 예상 소요 5~10분

**산출물 위치**: `MyDrive/morning-report/sector_data/` 아래 parquet 3개

---

## 섹션 0: 환경 준비

In [ ]:
# pykrx + pyarrow 설치 (Colab 기본 이미지에 없음)
!pip install -q pykrx pyarrow

In [ ]:
import sys
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from pykrx import stock

print(f'pandas {pd.__version__}')
print(f'python {sys.version.split()[0]}')

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/morning-report/sector_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'산출물 저장 경로: {OUTPUT_DIR}')

In [ ]:
# 기간: 최근 2년
END_DATE = datetime.now().strftime('%Y%m%d')
START_DATE = (datetime.now() - timedelta(days=365 * 2 + 10)).strftime('%Y%m%d')
print(f'수집 기간: {START_DATE} ~ {END_DATE}')

In [ ]:
# UNIVERSE (162종목) 로딩 — UZymn 브랜치에서 직접 clone (이미 있으면 pull)
import subprocess

REPO_DIR = Path('/content/morning-report')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '-b', 'claude/session-start-UZymn', '--depth', '1',
        'https://github.com/gengar200005/morning-report.git',
        str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', 'claude/session-start-UZymn'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'FETCH_HEAD'], check=True)

sys.path.insert(0, str(REPO_DIR / 'backtest'))
# 이미 import 되었으면 리로드
if 'universe' in sys.modules:
    import importlib
    import universe  # type: ignore
    importlib.reload(universe)
    from universe import UNIVERSE  # type: ignore
else:
    from universe import UNIVERSE  # type: ignore

print(f'UNIVERSE: {len(UNIVERSE)}종목')
print(f'처음 3: {UNIVERSE[:3]}')
print(f'마지막 3: {UNIVERSE[-3:]}')

UNIVERSE_TICKERS = [t for t, _ in UNIVERSE]
UNIVERSE_NAMES = dict(UNIVERSE)

---

## 섹션 1: KRX 22업종 분류 매핑

KOSPI 업종지수(약 22개) + KOSDAQ 섹터지수를 조회하고, 각 지수의 구성종목을
역매핑해서 **UNIVERSE 162종목 → 업종** 분류표를 만든다.

산출: `sector_map.parquet`
- 컬럼: `ticker, name, market, sector_ticker, sector_name`
- 매핑 실패 종목은 market='UNMAPPED'로 표시 (Phase C에서 overrides.yaml 대상)

In [ ]:
# 1-1. KOSPI + KOSDAQ 전체 지수 목록 수집
# pykrx가 현재 날짜에서 __fetch 실패하는 경우(transient or 장중 데이터 미확정)
# 대비: 과거 영업일로 폴백 + 재시도
import time as _time
from datetime import timedelta as _td

def _fetch_tickers(market: str):
    last_err = None
    for offset in [0, 1, 3, 7, 14]:
        try_date = (datetime.now() - _td(days=offset)).strftime('%Y%m%d')
        for attempt in range(3):
            try:
                tickers = stock.get_index_ticker_list(date=try_date, market=market)
                if tickers:
                    print(f'  [{market}] ref_date={try_date}: {len(tickers)}개 지수')
                    return tickers
                last_err = 'empty list'
            except Exception as e:
                last_err = f'{type(e).__name__}: {e}'
            _time.sleep(3)
    raise RuntimeError(f'{market} 지수 fetch 실패 (모든 후보 날짜/재시도): {last_err}')

def list_indices(market: str) -> pd.DataFrame:
    tickers = _fetch_tickers(market)
    rows = []
    for t in tickers:
        try:
            nm = stock.get_index_ticker_name(t)
        except Exception as e:
            nm = f'ERR:{e}'
        rows.append({'sector_ticker': t, 'sector_name': nm, 'market': market})
    return pd.DataFrame(rows)

kospi_idx = list_indices('KOSPI')
kosdaq_idx = list_indices('KOSDAQ')
all_idx = pd.concat([kospi_idx, kosdaq_idx], ignore_index=True)
print(f'\nKOSPI 지수: {len(kospi_idx)}, KOSDAQ 지수: {len(kosdaq_idx)}, 합: {len(all_idx)}')
all_idx

In [ ]:
# 1-2. 업종지수만 필터 — 규모별/우량/테마/종합지수 제외
# KOSPI:  코스피(1001), 대형주/중형주/소형주(1002~1004), 배당/섹터 ETF, 코스피200 계열 제외
#         → 업종은 '코스피 ' prefix + 분야명 (음식료품, 전기전자 등). 일반적으로 1005~1080 대역.
# KOSDAQ: 코스닥 전체지수/대형/중형/소형 제외 → 기타서비스/IT/제조 등 업종.
EXCLUDE_PATTERNS = [
    '코스피$', '코스피 200', '코스피 100', '코스피 50',
    '대형주', '중형주', '소형주', '배당', '우선주',
    '코스닥$', '코스닥 ?150', '코스닥 ?프리미어',
    'KRX ', 'ESG', '탄소', '레버리지', '인버스',
]
import re
exclude_re = re.compile('|'.join(EXCLUDE_PATTERNS))

sector_candidates = all_idx[~all_idx['sector_name'].astype(str).str.contains(exclude_re, na=False)].copy()
print(f'업종지수 후보: {len(sector_candidates)}개')
sector_candidates

In [ ]:
# 1-3. 각 업종지수의 구성종목 수집 → UNIVERSE 교차 매핑
#      한 종목이 여러 지수에 포함될 수 있음 (예: 코스피 제조업 = 여러 업종 aggregate).
#      → UNIVERSE 종목별로 '가장 구체적인' 업종 하나 선택: 구성종목 수가 가장 적은 지수 = 가장 세분류.
from collections import defaultdict

universe_set = set(UNIVERSE_TICKERS)
sector_size = {}
stock_to_sectors = defaultdict(list)

for _, row in sector_candidates.iterrows():
    sec_t = row['sector_ticker']
    sec_n = row['sector_name']
    try:
        members = stock.get_index_portfolio_deposit_file(sec_t, END_DATE)
        members = list(members) if not isinstance(members, pd.DataFrame) else list(members.index)
    except Exception as e:
        print(f'  [!] {sec_t} {sec_n} 실패: {e}')
        continue
    sector_size[sec_t] = len(members)
    for m in members:
        if m in universe_set:
            stock_to_sectors[m].append((sec_t, sec_n, row['market']))

print(f'업종지수 구성종목 수집 완료: {len(sector_size)}개 지수')
print(f'UNIVERSE 중 최소 1개 업종 매칭: {len(stock_to_sectors)}/{len(UNIVERSE)}')

In [ ]:
# 1-4. 종목별 '대표 업종' 선택 — 구성종목 수 최소 = 가장 세분류 업종
rows = []
for ticker, nm in UNIVERSE:
    matches = stock_to_sectors.get(ticker, [])
    if not matches:
        rows.append({
            'ticker': ticker, 'name': nm, 'market': 'UNMAPPED',
            'sector_ticker': None, 'sector_name': None,
        })
        continue
    # 가장 세분류 선택 (구성종목 수 최소)
    best = min(matches, key=lambda x: sector_size.get(x[0], 10_000))
    rows.append({
        'ticker': ticker, 'name': nm, 'market': best[2],
        'sector_ticker': best[0], 'sector_name': best[1],
    })

sector_map = pd.DataFrame(rows)
print(f'전체 {len(sector_map)}종목 매핑 완료')
print(f'  매핑됨:    {(sector_map["market"] != "UNMAPPED").sum()}')
print(f'  UNMAPPED: {(sector_map["market"] == "UNMAPPED").sum()}')
print()
print('--- 섹터별 분포 (top 20) ---')
print(sector_map.groupby(['market', 'sector_name']).size().sort_values(ascending=False).head(20))
print()
print('--- UNMAPPED 종목 ---')
unmapped = sector_map[sector_map['market'] == 'UNMAPPED']
print(unmapped[['ticker', 'name']].to_string(index=False) if len(unmapped) else '(없음)')

In [ ]:
# 1-5. parquet 저장
out_path = OUTPUT_DIR / 'sector_map.parquet'
sector_map.to_parquet(out_path, index=False)
print(f'저장 완료: {out_path}')
print(f'파일 크기: {out_path.stat().st_size / 1024:.1f} KB')

---

## 섹션 2: 업종지수 주봉 OHLCV

섹션 1에서 실제로 매핑에 사용된 업종지수들의 **일봉** OHLCV를 수집하고,
금요일 기준(W-FRI) **주봉**으로 리샘플한다.

왜 주봉?
- **Weinstein Stage 2**는 본래 주봉 차트 + MA30주 기반 (Stan Weinstein, 1988)
- **IBD 6M Relative Strength**는 주/일봉 무관 (6개월 수익률은 동일)
- 주봉으로 통일 시 노이즈 감소 + 데이터 크기 ~1/5

산출: `sector_index_weekly.parquet`
- 컬럼: `날짜, sector_ticker, sector_name, market, 시가, 고가, 저가, 종가, 거래량, 거래대금`

In [ ]:
# 2-1. 수집 대상 업종지수 목록 — sector_map에서 실제 매핑에 사용된 것만
target_sectors = (
    sector_map[sector_map['sector_ticker'].notna()]
    [['sector_ticker', 'sector_name', 'market']]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f'수집 대상 업종지수: {len(target_sectors)}개')
target_sectors

In [ ]:
# 2-2. 각 업종지수의 일봉 OHLCV 수집 (2년치)
import time

daily_frames = []
fail_list = []
for i, row in target_sectors.iterrows():
    sec_t = row['sector_ticker']
    sec_n = row['sector_name']
    try:
        df = stock.get_index_ohlcv_by_date(START_DATE, END_DATE, sec_t)
    except Exception as e:
        fail_list.append((sec_t, sec_n, str(e)))
        continue
    if df is None or df.empty:
        fail_list.append((sec_t, sec_n, 'empty'))
        continue
    df = df.reset_index()
    df['sector_ticker'] = sec_t
    df['sector_name'] = sec_n
    df['market'] = row['market']
    daily_frames.append(df)
    if (i + 1) % 5 == 0:
        print(f'  진행: {i + 1}/{len(target_sectors)}')
    time.sleep(0.1)  # rate limit 예방

sector_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else pd.DataFrame()
print(f'\n일봉 수집 완료: {len(sector_daily)}행, {sector_daily["sector_ticker"].nunique() if len(sector_daily) else 0}개 업종')
if fail_list:
    print(f'실패 {len(fail_list)}개: {fail_list}')

In [ ]:
# 2-3. 일봉 → 주봉 (W-FRI) 리샘플
# pykrx 컬럼: 시가/고가/저가/종가/거래량/거래대금 (거래대금 없는 경우 대비 defensive)
sector_daily['날짜'] = pd.to_datetime(sector_daily['날짜'])

agg_cols = {
    '시가': 'first',
    '고가': 'max',
    '저가': 'min',
    '종가': 'last',
    '거래량': 'sum',
}
if '거래대금' in sector_daily.columns:
    agg_cols['거래대금'] = 'sum'

weekly_frames = []
for sec_t, grp in sector_daily.groupby('sector_ticker'):
    g = grp.set_index('날짜').sort_index()
    w = g[list(agg_cols.keys())].resample('W-FRI').agg(agg_cols).dropna(subset=['종가'])
    w['sector_ticker'] = sec_t
    w['sector_name'] = grp['sector_name'].iloc[0]
    w['market'] = grp['market'].iloc[0]
    weekly_frames.append(w.reset_index())

sector_weekly = pd.concat(weekly_frames, ignore_index=True)
print(f'주봉 변환 완료: {len(sector_weekly)}행, {sector_weekly["sector_ticker"].nunique()}개 업종')
print(f'기간: {sector_weekly["날짜"].min().date()} ~ {sector_weekly["날짜"].max().date()}')
sector_weekly.head()

In [ ]:
# 2-4. parquet 저장
out_path = OUTPUT_DIR / 'sector_index_weekly.parquet'
sector_weekly.to_parquet(out_path, index=False)
print(f'저장: {out_path}')
print(f'파일 크기: {out_path.stat().st_size / 1024:.1f} KB')

---

## 섹션 3: 162종목 일봉 OHLCV

UNIVERSE 162종목의 **일봉** OHLCV (최근 2년). Breadth 계산(% above MA50)은
일봉 기준이므로 종목 데이터는 일봉으로 유지한다.

산출: `stocks_daily.parquet`
- 컬럼: `날짜, ticker, name, 시가, 고가, 저가, 종가, 거래량, 거래대금, 등락률`
- 약 162 × 500일 ≈ 80,000행 예상

In [ ]:
# 3-1. 162종목 일봉 OHLCV 수집 (rate limit: 0.2초/호출 = 약 35초)
stock_frames = []
stock_fails = []
for i, (ticker, nm) in enumerate(UNIVERSE):
    try:
        df = stock.get_market_ohlcv_by_date(START_DATE, END_DATE, ticker)
    except Exception as e:
        stock_fails.append((ticker, nm, str(e)))
        continue
    if df is None or df.empty:
        stock_fails.append((ticker, nm, 'empty'))
        continue
    df = df.reset_index()
    df['ticker'] = ticker
    df['name'] = nm
    stock_frames.append(df)
    if (i + 1) % 20 == 0:
        print(f'  진행: {i + 1}/{len(UNIVERSE)}')
    time.sleep(0.2)

stocks_daily = pd.concat(stock_frames, ignore_index=True)
print(f'\n종목 일봉 수집 완료: {len(stocks_daily):,}행, {stocks_daily["ticker"].nunique()}종목')
print(f'기간: {pd.to_datetime(stocks_daily["날짜"]).min().date()} ~ {pd.to_datetime(stocks_daily["날짜"]).max().date()}')
if stock_fails:
    print(f'\n실패 {len(stock_fails)}종목:')
    for t, n, e in stock_fails[:10]:
        print(f'  {t} {n}: {e}')
    if len(stock_fails) > 10:
        print(f'  ... 외 {len(stock_fails) - 10}개')

In [ ]:
# 3-2. parquet 저장
stocks_daily['날짜'] = pd.to_datetime(stocks_daily['날짜'])
out_path = OUTPUT_DIR / 'stocks_daily.parquet'
stocks_daily.to_parquet(out_path, index=False)
print(f'저장: {out_path}')
print(f'파일 크기: {out_path.stat().st_size / 1024 / 1024:.2f} MB')

---

## 섹션 4: 완료 확인

산출물 3개 확인 + 용량 표시. 이 셀까지 정상 실행되면 Phase B 완료.

In [ ]:
# 4. 산출물 검증
expected = ['sector_map.parquet', 'sector_index_weekly.parquet', 'stocks_daily.parquet']
print(f'{"파일":<32} {"크기":>10}  {"행":>10}  {"고유키":>10}')
print('-' * 70)
for fname in expected:
    p = OUTPUT_DIR / fname
    if not p.exists():
        print(f'{fname:<32} [MISSING]')
        continue
    size_kb = p.stat().st_size / 1024
    df = pd.read_parquet(p)
    if fname == 'sector_map.parquet':
        key = f'{df["ticker"].nunique()}종목'
    elif fname == 'sector_index_weekly.parquet':
        key = f'{df["sector_ticker"].nunique()}업종'
    else:
        key = f'{df["ticker"].nunique()}종목'
    print(f'{fname:<32} {size_kb:>8.1f} KB  {len(df):>10,}  {key:>10}')

print('\n[OK] Phase B 완료 — Drive에서 parquet 3개 확인하고 Claude 에게 알려주세요.')